# Post-process computed POMDP policy

In [ ]:
import numpy as np

data         = np.load('alpha_vectors.npz', allow_pickle=True)
alpha        = data['alpha_vectors']   # object array (T,); alpha[t] is (n_t, S)
actions      = data['actions']         # object array (T,); actions[t] is (n_t,) int
action_names = data['action_names']    # (A,) str

T = len(alpha)
print(f'Horizon T={T}')
for t in range(T):
    print(f'  t={t}: {alpha[t].shape[0]} alpha-vectors, {alpha[t].shape[1]} states')

## Inspect alpha-vectors per timestep

In [ ]:
for t in range(T):
    print(f'--- t={t} ---')
    for v, a in zip(alpha[t], actions[t]):
        print(f'  {action_names[a]:25s}  {v}')

## Query optimal action for a given belief

In [ ]:
def optimal_action(t, belief):
    """Return the optimal action name and value at timestep t for a belief vector."""
    values = alpha[t] @ belief
    best   = np.argmax(values)
    return action_names[actions[t][best]], values[best]

# Example: at t=1, equal probability on states 2 and 3 (short vs long pile, stage 2)
b = np.array([0., 0., 0.5, 0.5])
for t in range(T):
    act, val = optimal_action(t, b)
    print(f't={t}, belief={b}  ->  {act}  (V={val:.1f})')

## Plot value function over 1D belief slice

Sweep `p = P(long pile)` from 0 to 1, keeping belief on the two relevant states.

In [ ]:
import matplotlib.pyplot as plt

p = np.linspace(0, 1, 200)

# State index pairs per timestep: (short_state, long_state)
state_pairs = {0: (0, 1), 1: (2, 3)}

fig, axes = plt.subplots(1, T, figsize=(6 * T, 4), sharey=False)
if T == 1:
    axes = [axes]

colors = {
    'order_short_no_info': '#0072B2',
    'order_short_drill':   '#009E73',
    'order_short_sonic':   '#E69F00',
    'order_long_no_info':  '#56B4E9',
    'order_long_drill':    '#CC79A7',
    'order_long_sonic':    '#D55E00',
}

for t, ax in enumerate(axes):
    s_short, s_long = state_pairs[t]
    S = alpha[t].shape[1]

    for v, a in zip(alpha[t], actions[t]):
        name = action_names[a]
        # Build belief: concentrate on the two relevant states
        beliefs = np.zeros((len(p), S))
        beliefs[:, s_short] = 1 - p
        beliefs[:, s_long]  = p
        values = beliefs @ v
        ax.plot(p, values, color=colors.get(name, 'black'), label=name)

    ax.set_title(f't={t}')
    ax.set_xlabel('P(long pile)')
    ax.set_ylabel('Value')
    ax.grid(True, alpha=0.3)

# Deduplicate legend
handles, labels = axes[0].get_legend_handles_labels()
by_label = dict(zip(labels, handles))
fig.legend(by_label.values(), by_label.keys(),
           loc='lower center', ncol=3, frameon=False,
           bbox_to_anchor=(0.5, -0.12))
plt.tight_layout()
plt.show()